In [ ]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
class PINNmodel(nn.Module):
    def __init__(self):
        super(PINNmodel, self).__init__()
        #skip transformation
        self.skip = nn.Linear(4, 5)
        #input layer
        self.LT1 = nn.Linear(4, 128)
        self.BN1 = nn.BatchNorm1d(128)
        #hidden 1
        self.LT2 = nn.Linear(128, 256)
        self.BN2 = nn.BatchNorm1d(256)
        #hidden 2
        self.LT3 = nn.Linear(256, 256)
        self.BN3 = nn.BatchNorm1d(256)
        #hidden 3
        self.LT4 = nn.Linear(256, 256)
        self.BN4 = nn.BatchNorm1d(256)
        #hidden 4
        self.LT5 = nn.Linear(256, 256)
        self.BN5 = nn.BatchNorm1d(256)
        #hidden 5
        self.LT6 = nn.Linear(256, 256)
        self.BN6 = nn.BatchNorm1d(256)
        #hidden 6
        self.LT7 = nn.Linear(256, 128)
        self.BN7 = nn.BatchNorm1d(128)
        #output layer
        self.LT8 = nn.Linear(128, 5)

    def _initialize_weights(self):
        # Apply Xavier initialization to all layers with weights
        for m in self.modules():
            if isinstance(m, nn.Linear):
                # Xavier normal initialization
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)  # Initialize biases to zero

    def forward(self, x):
        #skip
        skip = self.skip(x)
        #input layer
        x = self.LT1(x)
        x = self.BN1(x)
        x = nn.Tanh()(x)
        #hidden 1
        x = self.LT2(x)
        x = self.BN2(x)
        x = nn.Tanh()(x)
        #hidden 2
        x = self.LT3(x)
        x = self.BN3(x)
        x = nn.Tanh()(x)
        #hidden 3
        x = self.LT4(x)
        x = self.BN4(x)
        x = nn.Tanh()(x)
        #hidden 4
        x = self.LT5(x)
        x = self.BN5(x)
        x = nn.Tanh()(x)
        #hidden 5
        x = self.LT6(x)
        x = self.BN6(x)
        x = nn.Tanh()(x)
        #hidden 6
        x = self.LT7(x)
        x = self.BN7(x)
        x = nn.Tanh()(x)
        #output layer
        x = self.LT8(x)
        #joint
        x = skip + x

        return x

In [ ]:
#load and assemble input data
#extract the data from excel
def read_data(excel_name, column):
    df = pd.read_excel(excel_name, usecols=[column])
    df = np.array(df)
    return df

#randomize the dataset
def unison_shuffled_copies(a, b, c, d, e):
    p = np.random.permutation(len(a))
    return a[p], b[p], c[p], d[p], e[p]

route = '/content/drive/MyDrive/Colab Notebooks/N16S159.xlsx'
#route = 'N16S159.xlsx'
x = read_data(route, 0) #shape expected to be (num. of data, 1) for learning
y = read_data(route, 1) #shape expected to be (num. of data, 1) for learning
z = read_data(route, 2) #shape expected to be (num. of data, 1) for learning
t = read_data(route, 3) #shape expected to be (num. of data, 1) for learning
u = read_data(route, 4) #x velocity
v = read_data(route, 5) #y velocity
w = read_data(route, 6) #z velocity

#data post-treat
u = u[:, 0] #shape expected to be (num. of data, ) for learning
v = v[:, 0] #shape expected to be (num. of data, ) for learning
w = w[:, 0] #shape expected to be (num. of data, ) for learning

print(f'shape check: x: {x.shape}, y: {y.shape}, z: {z.shape}, t: {t.shape}, u: {u.shape}, v: {v.shape}, w: {w.shape}')

N_z = 0.003
N_x = 0.001
z_PDE = []
x_PDE = []
y_PDE = []
t_PDE = []
for i in range(51): #z
    for j in range(46): #x
            z_content = []
            y_content = []
            x_content = []
            t_content = []
            decide = 1
            for index in range(len(z)):
                if i*N_z == z[index] and j*N_x == x[index] :
                    decide = 0
            if decide == 1:
                z_content.append(i*N_z)
                x_content.append(j*N_x)
                y_content.append(0)
                t_content.append(10)
                z_PDE.append(z_content)
                x_PDE.append(x_content)
                y_PDE.append(y_content)
                t_PDE.append(t_content)
z_PDE = np.array(z_PDE)
x_PDE = np.array(x_PDE)
y_PDE = np.array(y_PDE)
t_PDE = np.array(t_PDE)
print(f'x_PDE: {x_PDE.shape}, y_PDE: {y_PDE.shape}, z_PDE: {z_PDE.shape}, t_PDE: {t_PDE.shape}')

N_labled = len(u)

z = np.concatenate((z, z_PDE))
y = np.concatenate((y, y_PDE))
x = np.concatenate((x, x_PDE))
t = np.concatenate((t, t_PDE))
print(f'x_total: {x.shape}, y_total: {y.shape}, z_total: {z.shape}, t_total: {t.shape}')
#training dataset
x_total = torch.tensor(x, dtype=torch.float32, requires_grad=True)
y_total = torch.tensor(y, dtype=torch.float32, requires_grad=True)
z_total = torch.tensor(z, dtype=torch.float32, requires_grad=True)
t_total = torch.tensor(t, dtype=torch.float32, requires_grad=True)

stack = torch.hstack((x_total, y_total, z_total, t_total))
#evaluation dataset
x_eval = torch.tensor(x, dtype=torch.float32, requires_grad=True)
y_eval = torch.tensor(y, dtype=torch.float32, requires_grad=True)
z_eval = torch.tensor(z, dtype=torch.float32, requires_grad=True)
t_eval = torch.tensor(t, dtype=torch.float32, requires_grad=True)

u = torch.tensor(u, dtype=torch.float32, requires_grad=True)
v = torch.tensor(v, dtype=torch.float32, requires_grad=True)
w = torch.tensor(w, dtype=torch.float32, requires_grad=True)

stack_eval = torch.hstack((x_eval, y_eval, z_eval, t_eval))

In [ ]:
#module for N-S euqtions and loss evaluation
model = PINNmodel()
model.to('cuda')
#model.train()

In [ ]:
def softplus(x):
    return torch.log(1 + torch.exp(x))
null = torch.zeros(x_total.shape[0], x_total.shape[0])
mse = nn.MSELoss()
def r2_score(y_true, y_pred):
    # Mean of the true values
    y_mean = torch.mean(y_true)
    # Residual sum of squares (SS_res)
    ss_res = torch.sum((y_true - y_pred) ** 2)
    # Total sum of squares (SS_tot)
    ss_tot = torch.sum((y_true - y_mean) ** 2)
    # R^2 score
    r2 = 1 - ss_res / ss_tot
    return r2

In [ ]:
def function(stack):
    model.train()
    output = model(stack.to('cuda'))
    output = output.to('cpu')
    u, v, w, p, rho = output[:, 0], output[:, 1], output[:, 2], softplus(output[:, 3]), softplus(output[:, 4])

    g = -9.81
    mu = 1.834e-05

    rho_t = torch.autograd.grad(rho, t_total, grad_outputs=torch.ones_like(rho), create_graph=True)[0]
    rho_x = torch.autograd.grad(rho, x_total, grad_outputs=torch.ones_like(rho), create_graph=True)[0]
    rho_y = torch.autograd.grad(rho, y_total, grad_outputs=torch.ones_like(rho), create_graph=True)[0]
    rho_z = torch.autograd.grad(rho, z_total, grad_outputs=torch.ones_like(rho), create_graph=True)[0]

    u_t = torch.autograd.grad(u, t_total, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_x = torch.autograd.grad(u, x_total, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_y = torch.autograd.grad(u, y_total, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_z = torch.autograd.grad(u, z_total, grad_outputs=torch.ones_like(u), create_graph=True)[0]

    v_t = torch.autograd.grad(v, t_total, grad_outputs=torch.ones_like(v), create_graph=True)[0]
    v_x = torch.autograd.grad(v, x_total, grad_outputs=torch.ones_like(v), create_graph=True)[0]
    v_y = torch.autograd.grad(v, y_total, grad_outputs=torch.ones_like(v), create_graph=True)[0]
    v_z = torch.autograd.grad(v, z_total, grad_outputs=torch.ones_like(v), create_graph=True)[0]

    w_t = torch.autograd.grad(w, t_total, grad_outputs=torch.ones_like(w), create_graph=True)[0]
    w_x = torch.autograd.grad(w, x_total, grad_outputs=torch.ones_like(w), create_graph=True)[0]
    w_y = torch.autograd.grad(w, y_total, grad_outputs=torch.ones_like(w), create_graph=True)[0]
    w_z = torch.autograd.grad(w, z_total, grad_outputs=torch.ones_like(w), create_graph=True)[0]

    p_x = torch.autograd.grad(p, x_total, grad_outputs=torch.ones_like(p), create_graph=True)[0]
    p_y = torch.autograd.grad(p, y_total, grad_outputs=torch.ones_like(p), create_graph=True)[0]
    p_z = torch.autograd.grad(p, z_total, grad_outputs=torch.ones_like(p), create_graph=True)[0]

    u_xx = torch.autograd.grad(u_x, x_total, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    u_yy = torch.autograd.grad(u_y, y_total, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
    u_zz = torch.autograd.grad(u_z, z_total, grad_outputs=torch.ones_like(u_z), create_graph=True)[0]

    v_xx = torch.autograd.grad(v_x, x_total, grad_outputs=torch.ones_like(v_x), create_graph=True)[0]
    v_yy = torch.autograd.grad(v_y, y_total, grad_outputs=torch.ones_like(v_y), create_graph=True)[0]
    v_zz = torch.autograd.grad(v_z, z_total, grad_outputs=torch.ones_like(v_z), create_graph=True)[0]

    w_xx = torch.autograd.grad(w_x, x_total, grad_outputs=torch.ones_like(w_x), create_graph=True)[0]
    w_yy = torch.autograd.grad(w_y, y_total, grad_outputs=torch.ones_like(w_y), create_graph=True)[0]
    w_zz = torch.autograd.grad(w_z, z_total, grad_outputs=torch.ones_like(w_z), create_graph=True)[0]

    #N-S
    F = rho*u_t + u*rho_t + u*(rho*u_x + u*rho_x) + v*(rho*u_y + u*rho_y) + w*(rho*u_z + u*rho_z) + p_x - mu*(u_xx + u_yy + u_zz)
    G = rho*v_t + v*rho_t + u*(rho*v_x + v*rho_x) + v*(rho*v_y + v*rho_y) + w*(rho*v_z + v*rho_z) + p_y - mu*(v_xx + v_yy + v_zz)
    H = rho*w_t + w*rho_t + u*(rho*w_x + w*rho_x) + v*(rho*w_y + w*rho_y) + w*(rho*w_z + w*rho_z) + p_z - mu*(w_xx + w_yy + w_zz) - rho*g
    #continuity
    I = rho_t + rho*(u_x + v_y + w_z) + u*rho_x + v*rho_y + w*rho_z

    return u, v, w, F, G, H, I

In [ ]:
def evaluation(stack):
    #model.eval()
    output = model(stack.to('cuda'))
    output = output.to('cpu')
    u, v, w = output[:, 0], output[:, 1], output[:, 2]
    return u, v, w

In [ ]:
optimizer1 = torch.optim.Adam(model.parameters(), lr=0.01)
for epoch in range(7000):
    #initialize
    if (epoch+1)  == 1500:
        optimizer1 = torch.optim.Adam(model.parameters(), lr=0.001)
    if (epoch+1)  == 3000:
        optimizer1 = torch.optim.Adam(model.parameters(), lr=0.0001)
    if (epoch+1)  == 5000:
        optimizer1 = torch.optim.Adam(model.parameters(), lr=0.00001)
    optimizer1.zero_grad()
    #forward
    u_pred, v_pred, w_pred, F_pred, G_pred, H_pred, I_pred= function(stack)

    #loss
    u_loss = mse(u_pred[0:N_labled], u) #assume that the first self.N_labled u v data are experimental measurement
    v_loss = mse(v_pred[0:N_labled], v)
    w_loss = mse(w_pred[0:N_labled], w)
    f_loss = mse(F_pred, null)
    g_loss = mse(G_pred, null)
    h_loss = mse(H_pred, null)
    I_loss = mse(I_pred, null)
    ls = 100*(u_loss + v_loss + w_loss) + (1)*(f_loss + g_loss + h_loss + I_loss)

    #loss grad
    ls.backward()
    #optimizing
    optimizer1.step()

    u_eval, v_eval, w_eval = evaluation(stack_eval)

    train_score = 1/3*(r2_score(u, u_pred[0:N_labled]) + r2_score(v, v_pred[0:N_labled]) + r2_score(w, w_pred[0:N_labled]))
    test_score = 1/3*(r2_score(u, u_eval[0:N_labled]) + r2_score(v, v_eval[0:N_labled]) + r2_score(w, w_eval[0:N_labled]))
    print(f'epoch: {epoch+1}, tol_loss: {ls:.2f}, PDE_loss: {1*(f_loss + g_loss + h_loss + I_loss):.2f}, labled_loss: {1*(u_loss + v_loss + w_loss):.2f}, ls: {optimizer1.param_groups[0]["lr"]}, training R2: {train_score}, testing R2: {test_score}')

In [ ]:
##saving the trained model
#torch.save(model.state_dict(), 'N16S159_best_0919.pt')

In [ ]:
##create nodes for a finer flow field
#N_z = 0.0005
#N_x = 0.0005
#z_PDE = []
#x_PDE = []
#y_PDE = []
#t_PDE = []
#for i in range(301): #z
#    for j in range(91): #x
#            z_content = []
#            y_content = []
#            x_content = []
#            t_content = []
#            decide = 1
#
#            if decide == 1:
#                z_content.append(i*N_z)
#                x_content.append(j*N_x)
#                y_content.append(0)
#                t_content.append(10)
#                z_PDE.append(z_content)
#                x_PDE.append(x_content)
#                y_PDE.append(y_content)
#                t_PDE.append(t_content)
#z_PDE = np.array(z_PDE)
#x_PDE = np.array(x_PDE)
#y_PDE = np.array(y_PDE)
#t_PDE = np.array(t_PDE)
#print(f'x_PDE: {x_PDE.shape}, y_PDE: {y_PDE.shape}, z_PDE: {z_PDE.shape}, t_PDE: {t_PDE.shape}')

In [ ]:
#evaluation dataset
x_eval = torch.tensor(x_PDE, dtype=torch.float32, requires_grad=True)
y_eval = torch.tensor(y_PDE, dtype=torch.float32, requires_grad=True)
z_eval = torch.tensor(z_PDE, dtype=torch.float32, requires_grad=True)
t_eval = torch.tensor(t_PDE, dtype=torch.float32, requires_grad=True)

stack_eval = torch.hstack((x_eval, y_eval, z_eval, t_eval))

In [ ]:
model = PINNmodel().to('cuda')
model.load_state_dict(torch.load('N16S159_best_0919.pt'))
u_eval, v_eval, w_eval = evaluation(stack_eval.to('cuda'))
u_eval = u_eval.to('cpu')
v_eval = v_eval.to('cpu')
w_eval = w_eval.to('cpu')
print(u_eval.shape)

In [ ]:
x_eval = x_eval.detach().numpy()
y_eval = y_eval.detach().numpy()
z_eval = z_eval.detach().numpy()
t_eval = t_eval.detach().numpy()
u_eval = u_eval.detach().numpy()
v_eval = v_eval.detach().numpy()
w_eval = w_eval.detach().numpy()

with open('N16S159_best_0919_data.dat', 'a') as w:
    for i in range(len(z_eval)):
        w.write(f'{x_eval[i][0]}  {y_eval[i][0]}    {z_eval[i][0]}    {np.sqrt((u_eval[i])**2 + (v_eval[i])**2 + (w_eval[i])**2)} {u_eval[i]} {v_eval[i]} {w_eval[i]}')
        w.write('\n')